# Football Player Tracker — Baseline Training (Colab)

Entrena YOLOv11n durante **30 épocas** en una T4 de Colab y loguea **en tiempo real** al MLflow que tienes corriendo en tu Mac, via túnel ngrok.

## Prerrequisitos (haz esto antes de abrir el notebook)

1. **Repo en GitHub.** Si todavía no lo subiste, corre en tu Mac:
   ```bash
   cd ~/Documents/Repositorios_Github/football-player-tracker
   git remote add origin https://github.com/TU_USUARIO/football-player-tracker.git
   git push -u origin main
   ```
2. **MLflow corriendo con SQLite backend:**
   ```bash
   mlflow ui --backend-store-uri sqlite:///mlflow.db --host 0.0.0.0 --port 8080
   ```
3. **ngrok exponiendo el puerto 8080:**
   ```bash
   ngrok http --host-header=rewrite 8080
   ```
4. **Colab Secrets** configurados (ícono 🔑 en el panel izquierdo de Colab):
   - `GITHUB_REPO_URL` — URL HTTPS del repo, ej. `https://github.com/jmvazqueznicolas/football-player-tracker.git`
   - `MLFLOW_NGROK_URL` — La URL que te dio ngrok, ej. `https://aea2-xxxx.ngrok-free.app`
   - `ROBOFLOW_API_KEY` — Tu API key de Roboflow

## Orden de ejecución
Corre las celdas **en orden**. Solo la celda de configuración requiere tu atención; el resto es automático.

## 0. Verificar GPU

In [ ]:
import subprocess

result = subprocess.run(["nvidia-smi"], capture_output=True, text=True)
print(
    result.stdout
    if result.returncode == 0
    else "⚠️  No GPU detectada — actívala en Entorno de ejecución → Cambiar tipo de entorno de ejecución → T4 GPU"
)

## 1. Leer secrets de Colab

Los valores viven en el panel 🔑 de Colab (nunca en el código).

In [ ]:
from google.colab import userdata

GITHUB_REPO_URL = userdata.get(
    "GITHUB_REPO_URL"
)  # ej. https://github.com/user/football-player-tracker.git
MLFLOW_NGROK_URL = userdata.get("MLFLOW_NGROK_URL")  # ej. https://aea2-xxxx.ngrok-free.app
ROBOFLOW_API_KEY = userdata.get("ROBOFLOW_API_KEY")

missing = [
    k
    for k, v in {
        "GITHUB_REPO_URL": GITHUB_REPO_URL,
        "MLFLOW_NGROK_URL": MLFLOW_NGROK_URL,
        "ROBOFLOW_API_KEY": ROBOFLOW_API_KEY,
    }.items()
    if not v
]

if missing:
    raise OSError(
        f"Faltan secrets en Colab: {missing}. Agrégalos en el panel 🔑 y vuelve a correr."
    )

print("✓ Secrets cargados correctamente")
print(f"  Repo:   {GITHUB_REPO_URL}")
print(f"  MLflow: {MLFLOW_NGROK_URL}")

## 2. Instalar dependencias

Solo lo que necesita el pipeline de entrenamiento. No usamos Poetry en Colab — pip directo es más simple.

In [ ]:
!pip install -q \
    'ultralytics>=8.3' \
    'mlflow>=3.12.0' \
    'hydra-core>=1.3' \
    'omegaconf>=2.3' \
    'loguru>=0.7' \
    'python-dotenv>=1.0' \
    'pyyaml>=6.0' \
    'roboflow>=1.1'
print('✓ Dependencias instaladas')

## 3. Clonar el repositorio

In [ ]:
import os
from pathlib import Path

REPO_DIR = Path("/content/football-player-tracker")

if REPO_DIR.exists():
    print("Repo ya existe, haciendo pull...")
    !git -C {REPO_DIR} pull
else:
    !git clone {GITHUB_REPO_URL} {REPO_DIR}

os.chdir(REPO_DIR)
print(f"✓ Directorio de trabajo: {os.getcwd()}")

## 4. Descargar dataset desde Roboflow

Usa el mismo script que tenemos en el repo. Si el dataset ya existe en esta sesión de Colab, lo saltamos.

In [ ]:
DATASET_DIR = REPO_DIR / "data" / "processed" / "football-players"

if DATASET_DIR.exists():
    n_train = len(list((DATASET_DIR / "train" / "images").glob("*")))
    print(f"✓ Dataset ya presente ({n_train} imágenes en train). Saltando descarga.")
else:
    print("Descargando dataset desde Roboflow...")
    !python scripts/download_dataset.py --api-key {ROBOFLOW_API_KEY}
    print("✓ Dataset descargado")

## 5. Configurar variables de entorno

Le decimos al pipeline dónde está el MLflow (tu Mac via ngrok).

In [ ]:
import os

os.environ["MLFLOW_TRACKING_URI"] = MLFLOW_NGROK_URL
os.environ["ROBOFLOW_API_KEY"] = ROBOFLOW_API_KEY
os.environ["MLFLOW_EXPERIMENT_NAME"] = "football-player-tracker"

# Escribir .env para que python-dotenv lo pickee dentro del módulo de training
env_content = f"MLFLOW_TRACKING_URI={MLFLOW_NGROK_URL}\nROBOFLOW_API_KEY={ROBOFLOW_API_KEY}\nMLFLOW_EXPERIMENT_NAME=football-player-tracker\n"
(REPO_DIR / ".env").write_text(env_content)

print("✓ Variables de entorno configuradas")
print(f"  MLFLOW_TRACKING_URI = {MLFLOW_NGROK_URL}")

## 6. Verificar conectividad con MLflow

Antes de arrancar el entrenamiento, confirmamos que Colab puede hablar con tu MLflow local via ngrok.

In [ ]:
import mlflow

mlflow.set_tracking_uri(MLFLOW_NGROK_URL)

try:
    experiments = mlflow.search_experiments()
    print(f"✓ Conexión OK — {len(experiments)} experimento(s) en el servidor")
    for exp in experiments:
        print(f"  • {exp.name} (id={exp.experiment_id})")
except Exception as e:
    print(f"✗ No se pudo conectar al MLflow en {MLFLOW_NGROK_URL}")
    print(f"  Error: {e}")
    print()
    print("Checklist:")
    print("  1. ¿Está corriendo mlflow ui --backend-store-uri sqlite:///mlflow.db en tu Mac?")
    print("  2. ¿Está corriendo ngrok http --host-header=rewrite 8080?")
    print("  3. ¿Copiaste bien la URL ngrok (https://...) en el secret MLFLOW_NGROK_URL?")

## 7. Entrenamiento baseline

**30 épocas, YOLOv11n, imgsz=640, T4 GPU.**

Mientras corre, puedes abrir tu MLflow UI (`http://localhost:8080`) y ver las métricas actualizarse época a época.

Tiempo estimado en T4: ~25–35 minutos.

In [ ]:
# Registrar el paquete como importable sin resolver dependencias
# (--no-deps porque ya las instalamos en la celda 2 — evita 30 min de resolver)
!pip install -e . --no-deps -q

# device=0  → primera GPU (T4)
# batch=16  → default del config, T4 lo aguanta con imgsz=640
# workers=2 → Colab tiene CPUs limitados, más workers no ayuda
!python -m football_tracker.training.train \
    training.epochs=30 \
    training.device=0 \
    training.workers=2 \
    training.batch=16

## 8. Guardar artefactos

Copiamos `best.pt` a `models/baseline/` y mostramos las métricas finales del run.

In [ ]:
import glob
import shutil
from pathlib import Path

# best.pt está dentro del dir de salida de Hydra: runs/<timestamp>/weights/best.pt
candidates = sorted(
    glob.glob(str(REPO_DIR / "runs" / "**" / "weights" / "best.pt"), recursive=True)
)

if not candidates:
    print("⚠️  No se encontró best.pt. Revisa si el entrenamiento terminó correctamente.")
else:
    best_pt = Path(candidates[-1])
    dest_dir = REPO_DIR / "models" / "baseline"
    dest_dir.mkdir(parents=True, exist_ok=True)
    shutil.copy(best_pt, dest_dir / "best.pt")
    print(f'✓ best.pt copiado a: {dest_dir / "best.pt"}')
    print(f'  Tamaño: {(dest_dir / "best.pt").stat().st_size / 1e6:.1f} MB')

In [ ]:
# Métricas finales del último run de MLflow
import mlflow

mlflow.set_tracking_uri(MLFLOW_NGROK_URL)
experiment = mlflow.get_experiment_by_name("football-player-tracker")

if experiment:
    runs = mlflow.search_runs(
        experiment_ids=[experiment.experiment_id], order_by=["start_time DESC"], max_results=1
    )
    if not runs.empty:
        run = runs.iloc[0]
        print("=== Métricas finales del baseline ===")
        metric_cols = [c for c in run.index if c.startswith("metrics.")]
        for col in sorted(metric_cols):
            val = run[col]
            if val == val:  # skip NaN
                print(f'  {col.replace("metrics.", ""):40s} {val:.4f}')
    else:
        print("No se encontraron runs en el experimento.")
else:
    print("Experimento no encontrado en el servidor MLflow.")

## 9. Descargar best.pt a tu Mac

Colab no persiste archivos entre sesiones — descárgalo ahora.

In [ ]:
from google.colab import files

best_pt = REPO_DIR / "models" / "baseline" / "best.pt"
if best_pt.exists():
    files.download(str(best_pt))
    print("✓ Descarga iniciada.")
    print("  Guarda el archivo en models/baseline/ dentro de tu repo local.")
else:
    print("⚠️  No se encontró best.pt. Corre primero la celda de entrenamiento.")

## 10. Siguientes pasos

Una vez que tengas `best.pt` en tu Mac:

1. Cópialo a `models/baseline/best.pt` en tu repo local.
2. Verifica en tu MLflow UI (`http://localhost:8080`) que el run aparece en la pestaña **Models** con el alias `@candidate`.
3. Anota las métricas clave en el `README.md` bajo la sección **Results**.
4. Commit y push:
   ```bash
   git add models/baseline/best.pt README.md
   git commit -m 'feat: add 30-epoch baseline weights and results'
   git push
   ```
5. Con esto cerramos **Session 1**. La Session 2 arranca con integración de W&B en paralelo con MLflow.